# Memorie, dispozitive și cuantizare

De câtă memorie are nevoie un model de limbaj rulat local, cum îl punem pe placa grafică și cum îl micșorăm prin cuantizare pe 4 biți, cu Transformers și bitsandbytes.

Codul din acest notebook vine de pe pagina [Memorie, dispozitive și cuantizare](https://inteligentartificiala.ro/laboratoare/model-local-transformers/memorie-si-dispozitive/) a laboratorului [Un model de limbaj local cu Transformers](https://inteligentartificiala.ro/laboratoare/model-local-transformers/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro). Explicațiile fiecărui pas sunt pe pagină.

În Google Colab, codul rulează mult mai repede pe o placă grafică. Dacă nu este deja aleasă, se alege din meniul **Runtime**, opțiunea **Change runtime type**, o variantă cu GPU. Celulele se rulează în ordine, de sus în jos.

In [ ]:
%pip install -q -U transformers accelerate

## De câtă memorie are nevoie un model

Metoda `get_memory_footprint` spune cât ocupă ponderile unui model deja încărcat.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
for dtype in [torch.float32, torch.bfloat16]:
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype)
    print(f"{str(dtype):15s} {model.get_memory_footprint() / 1e6:6.0f} MB")

## Cuantizarea

**Cuantizarea** (_quantization_) stochează ponderile cu mai puțini biți, de exemplu pe 4 biți în loc de 16. Modelul ocupă de câteva ori mai puțină memorie și pierde, de obicei, puțin din calitate. Cea mai simplă metodă în Transformers folosește biblioteca **bitsandbytes**, care cuantizează ponderile chiar în momentul încărcării. Pentru a o instala, rulați:

In [ ]:
%pip install -q bitsandbytes

Codul de mai jos încarcă același model pe 4 biți, în formatul NF4.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M-Instruct", quantization_config=quantization, device_map="auto"
)
print(f"4 biți (NF4): {model.get_memory_footprint() / 1e6:.0f} MB")